# 05 — Fixed-Target Reliability Summary

Run independently: this notebook loads current completed DB models and champion
results itself. It exports **the fixed-target attainment PNG** plus native and
noise terminal-reliability CSVs. It does not compute adaptive AUC profiles, load
cross-function traces, run evaluations, or display images inline.

Detailed strategy/convergence profiles are in `06_convergence_ecdf_and_ablations.ipynb`;
noise robustness is in `07_noise_robustness.ipynb`. Each export is
cached separately and refreshed only when relevant inputs, settings or code change.


In [4]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bootstrap").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from bootstrap.analysis import build_analysis_use_cases, build_analysis_results_store
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter


results_store = build_analysis_results_store()
figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None



## Calculate reliability (no figure or report export)


In [5]:
with build_analysis_use_cases() as analysis:
    reliability = analysis.reliability.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
print(f"Calculated {len(reliability.native)} current native condition rows.")
display(reliability.native)

reliability_snapshot = results_store.save(reliability)
print(f"Saved numerical results: {reliability_snapshot}")


Calculated 8 current native condition rows.


,Model,Strategy,Solver,Source Problem,Problem ID,Dim,Noise Std,Code Hash,Trials,Complete,Primary Successes,Primary Success Rate,Secondary Success Rate,CI Lower,CI Upper,Median Terminal Error
0,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline_implicit,15,15,10,0.1,ad5e1e51f549543da060198f7229b01fbf20aa80c14394...,20,True,0,0.0,0.0,0.0,0.161125,31.227818
1,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline_noisy,15,15,10,0.1,f22401fa35d27086cc35e0446d84a0a2ad409e4b01835b...,20,True,0,0.0,0.0,0.0,0.161125,32.710980
2,qwen2.5-coder-32b-instruct-q4_k_m.gguf,guided,qwen_32b_guided_implicit,15,15,10,0.1,3ea798e25036ffafef79f5e66bdad27b50b935bfd73c08...,20,True,0,0.0,0.0,0.0,0.161125,15.694812
3,qwen2.5-coder-32b-instruct-q4_k_m.gguf,guided,qwen_32b_guided_noisy,15,15,10,0.1,4c2bf9c0945b25170141e8e2d97d77498e3e60adc167fa...,20,True,0,0.0,0.0,0.0,0.161125,10.743458
4,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,qwen_32b_thinking_implicit,15,15,10,0.1,a3ce8749512124dbdf8089a42460559debfe47a01ca183...,20,True,0,0.0,0.0,0.0,0.161125,51.750210
5,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,qwen_32b_thinking_noisy,15,15,10,0.1,07108a9fd99ef72a9e72b621069ac26fa633bb26f46112...,20,True,0,0.0,0.0,0.0,0.161125,45.959823
6,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,qwen_32b_vectorization_implicit,15,15,10,0.1,651dd578793124c4361f102d5ce23db4e910281ecf4af7...,20,True,0,0.0,0.0,0.0,0.161125,10.069508
7,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,qwen_32b_vectorization_noisy,15,15,10,0.1,ad1bce504df2c916fff63e2268bc6255ea017836bb7259...,20,True,0,0.0,0.0,0.0,0.161125,39.963411


Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/reliability/7e7f06e0daadaa8bd201440cfd0f148515230c33ec3b18a908e1f05605bca7f7


## Reliability tables and fixed-target attainment export

The matrix scores returned solutions. The attainment curve describes the best
clean point queried, which need not be returned. Noise reliability uses identical
frozen champions, not separately synthesized noisy champions. Incomplete conditions
are excluded from aggregate figures, not counted as failures.


In [6]:
figure_exporter.export_reliability(reliability)
